In [19]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [20]:
torch.manual_seed(42)

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
df = pd.read_csv('/content/drive/MyDrive/Dataset/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


Tokenization -> Vocab building -> converting Text to Numbers (Indexs)

In [24]:
def tokenize(text):
    text = text.lower()
    text = text.replace('?', '')
    text = text.replace("'", "")
    return text.split()

vocab = {'<UNK>': 0}

def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens = tokenized_question + tokenized_answer
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)

def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [35]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [36]:
import torch
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [37]:
dataset = QADataset(df, vocab)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [38]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[ 78,  79, 150, 151,  14, 152, 153]]) tensor([154])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([205])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([215])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([53])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([124])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([61])
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([220])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([298])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([54])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([173])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([238])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([285])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([128])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([268])
tensor([[  1,   2,   3, 163, 164, 165,  83, 

In [39]:
import torch.nn as nn

class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [43]:
learning_rate = 0.001
epochs = 35

In [44]:
model = SimpleRNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [45]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 523.463302
Epoch: 2, Loss: 457.929410
Epoch: 3, Loss: 381.714602
Epoch: 4, Loss: 317.352482
Epoch: 5, Loss: 264.749558
Epoch: 6, Loss: 215.875057
Epoch: 7, Loss: 171.991417
Epoch: 8, Loss: 133.677740
Epoch: 9, Loss: 101.705088
Epoch: 10, Loss: 77.980899
Epoch: 11, Loss: 60.109116
Epoch: 12, Loss: 47.003268
Epoch: 13, Loss: 37.472803
Epoch: 14, Loss: 30.779343
Epoch: 15, Loss: 25.590036
Epoch: 16, Loss: 21.576709
Epoch: 17, Loss: 18.383281
Epoch: 18, Loss: 15.995456
Epoch: 19, Loss: 13.886340
Epoch: 20, Loss: 11.984518
Epoch: 21, Loss: 10.473822
Epoch: 22, Loss: 9.164745
Epoch: 23, Loss: 8.123389
Epoch: 24, Loss: 7.195136
Epoch: 25, Loss: 6.401759
Epoch: 26, Loss: 5.742754
Epoch: 27, Loss: 5.201043
Epoch: 28, Loss: 4.680562
Epoch: 29, Loss: 4.275128
Epoch: 30, Loss: 3.899424
Epoch: 31, Loss: 3.563390
Epoch: 32, Loss: 3.274720
Epoch: 33, Loss: 3.018964
Epoch: 34, Loss: 2.794299
Epoch: 35, Loss: 2.578631


In [50]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

In [52]:
predict(model, "What is the largest planet in our solar system?")

jupiter
